In [50]:
import os
import re
import glob
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy

from sklearn.metrics import precision_recall_fscore_support, classification_report

In [51]:
# Semilla para que la selección aleatoria de oraciones sea reproducible.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Carga del modelo preentrenado de spaCy.
nlp = spacy.load("en_core_web_sm")

print("Modelo cargado correctamente.")
print("Versión de spaCy:", spacy.__version__)

Modelo cargado correctamente.
Versión de spaCy: 3.8.14


In [52]:
# Buscar archivos de texto y tablas dentro del entorno de trabajo.

DATASET_DIR = Path(".")

archivos = sorted(
    [
        p for p in DATASET_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in [".txt", ".csv", ".json", ".xlsx"]
    ]
)

print("Archivos encontrados:")
for archivo in archivos:
    print("-", archivo)

Archivos encontrados:
- gutenberg_novels_dataset.csv


In [53]:

archivos_csv = [p for p in archivos if p.suffix.lower() == ".csv"]

for archivo in archivos_csv:
    print(f"\nArchivo: {archivo}")
    
    df_inspeccion = pd.read_csv(archivo)
    
    print("Dimensiones:", df_inspeccion.shape)
    print("Columnas:", list(df_inspeccion.columns))
    display(df_inspeccion.head())


Archivo: gutenberg_novels_dataset.csv
Dimensiones: (3, 6)
Columnas: ['gutenberg_id', 'title', 'author', 'download_url', 'text_length', 'text']


,gutenberg_id,title,author,download_url,text_length,text
0,1342,Pride and Prejudice,Jane Austen,https://www.gutenberg.org/files/1342/1342-0.txt,728392,*** START OF THE PROJECT GUTENBERG EBOOK 1342 ...
1,84,Frankenstein,Mary Shelley,https://www.gutenberg.org/files/84/84-0.txt,419290,*** START OF THE PROJECT GUTENBERG EBOOK 84 **...
2,345,Dracula,Bram Stoker,https://www.gutenberg.org/files/345/345-0.txt,845805,*** START OF THE PROJECT GUTENBERG EBOOK 345 *...


In [54]:
# Cargar el dataset completo desde el archivo CSV.

archivo_dataset = archivos_csv[0]
df = pd.read_csv(archivo_dataset)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)
print("Columnas:", list(df.columns))

# Mostrar la información principal de cada novela.
display(df[["gutenberg_id", "title", "author", "text_length"]])

Dataset cargado correctamente.
Dimensiones: (3, 6)
Columnas: ['gutenberg_id', 'title', 'author', 'download_url', 'text_length', 'text']


,gutenberg_id,title,author,text_length
0,1342,Pride and Prejudice,Jane Austen,728392
1,84,Frankenstein,Mary Shelley,419290
2,345,Dracula,Bram Stoker,845805


In [55]:
# Verificar que existan tres novelas y que cada una tenga texto.

if len(df) != 3:
    print(f"Advertencia: se encontraron {len(df)} registros en lugar de 3.")
else:
    print("Se encontraron exactamente tres novelas.")

for i, fila in df.iterrows():
    print(f"\nNovela {i + 1}")
    print("Título:", fila["title"])
    print("Autor:", fila["author"])
    print("Longitud registrada:", fila["text_length"])
    print("Longitud real del texto:", len(str(fila["text"])))
    print("Inicio del texto:")
    print(str(fila["text"])[:300])

Se encontraron exactamente tres novelas.

Novela 1
Título: Pride and Prejudice
Autor: Jane Austen
Longitud registrada: 728392
Longitud real del texto: 728392
Inicio del texto:
*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***

                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKI

Novela 2
Título: Frankenstein
Autor: Mary Shelley
Longitud registrada: 419290
Longitud real del texto: 419290
Inicio del texto:
*** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley

 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chapt

Novela 3
Título: Dracula
Autor: Bram Stoker
Longitud registrada: 845805
Longitud real del te

In [56]:
def limpiar_gutenberg(texto):
    """
    Elimina el encabezado y el final estándar de Project Gutenberg.
    """
    texto = str(texto).replace("\r\n", "\n").replace("\r", "\n")
    
    # Eliminar el encabezado de Project Gutenberg.
    patron_inicio = r"\*\*\* START OF (?:THE )?PROJECT GUTENBERG EBOOK.*?\*\*\*"
    texto = re.sub(patron_inicio, "", texto, count=1, flags=re.IGNORECASE | re.DOTALL)
    
    # Eliminar el cierre de Project Gutenberg.
    patron_final = r"\*\*\* END OF (?:THE )?PROJECT GUTENBERG EBOOK.*"
    texto = re.sub(patron_final, "", texto, count=1, flags=re.IGNORECASE | re.DOTALL)
    
    # Normalizar espacios y saltos de línea.
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    
    return texto.strip()

In [57]:
# Preparar el texto completo de cada novela.
# Se conservan las palabras y los signos de puntuación porque spaCy
# los utilizará como tokens durante el etiquetado POS y NER.

novelas = {}

for _, fila in df.iterrows():
    titulo = fila["title"]
    texto_limpio = limpiar_gutenberg(fila["text"])
    
    novelas[titulo] = {
        "titulo": titulo,
        "autor": fila["author"],
        "id": fila["gutenberg_id"],
        "texto": texto_limpio
    }

print("Novelas preparadas:")
for titulo, datos in novelas.items():
    print(f"- {titulo}: {len(datos['texto']):,} caracteres")

Novelas preparadas:
- Pride and Prejudice: 721,387 caracteres
- Frankenstein: 419,141 caracteres
- Dracula: 840,765 caracteres


In [58]:
# Segmentar cada novela en oraciones y tokens.
# Se utiliza spaCy únicamente para obtener las oraciones en esta etapa.
# El etiquetado POS y NER se aplicará posteriormente sobre las muestras.

nlp_preparacion = spacy.load("en_core_web_sm")
nlp_preparacion.max_length = 2_000_000

for titulo, datos in novelas.items():
    doc = nlp_preparacion(datos["texto"])
    
    oraciones = [
        sent.text.strip()
        for sent in doc.sents
        if len(sent.text.strip()) > 0
    ]
    
    tokens_totales = [
        token.text
        for token in doc
        if not token.is_space
    ]
    
    datos["oraciones"] = oraciones
    datos["tokens_totales"] = tokens_totales
    datos["num_oraciones"] = len(oraciones)
    datos["num_tokens"] = len(tokens_totales)
    
    print(
        f"{titulo}: "
        f"{datos['num_oraciones']:,} oraciones, "
        f"{datos['num_tokens']:,} tokens"
    )

Pride and Prejudice: 5,754 oraciones, 152,543 tokens
Frankenstein: 3,249 oraciones, 85,881 tokens
Dracula: 8,452 oraciones, 191,530 tokens


In [59]:
# Seleccionar una muestra aleatoria de 100 oraciones por novela.
# La misma muestra se utilizará como base para las secciones de POS.

for titulo, datos in novelas.items():
    cantidad_muestra = min(100, len(datos["oraciones"]))
    
    datos["muestra_oraciones"] = random.sample(
        datos["oraciones"],
        cantidad_muestra
    )
    
    datos["muestra_texto"] = " ".join(datos["muestra_oraciones"])
    
    datos["muestra_tokens"] = [
        token.text
        for token in nlp(datos["muestra_texto"])
        if not token.is_space
    ]
    
    datos["num_muestra_oraciones"] = len(datos["muestra_oraciones"])
    datos["num_muestra_tokens"] = len(datos["muestra_tokens"])

In [60]:
# Crear la tabla comparativa solicitada en la sección 1.

tabla_preparacion = pd.DataFrame([
    {
        "Novela": datos["titulo"],
        "Autor": datos["autor"],
        "Oraciones completas": datos["num_oraciones"],
        "Tokens completos": datos["num_tokens"],
        "Oraciones de muestra": datos["num_muestra_oraciones"],
        "Tokens de muestra": datos["num_muestra_tokens"]
    }
    for datos in novelas.values()
])

display(tabla_preparacion)

,Novela,Autor,Oraciones completas,Tokens completos,Oraciones de muestra,Tokens de muestra
0,Pride and Prejudice,Jane Austen,5754,152543,100,2675
1,Frankenstein,Mary Shelley,3249,85881,100,2641
2,Dracula,Bram Stoker,8452,191530,100,2299
